# Create DOE OSTI Facility-Proposal Awards

**U.S. Department of Energy** (funder_id `4320306084`, priority `455`, provenance
`osti_facility`). Source: **OSTI's Crossref grant-DOI registrations** (Crossref member 960,
`type:grant`; see scripts/local/osti_facility_to_s3.py). **17,770 facility-proposal records /
17,067 distinct award numbers / 39 facility codes** (EMSL, JGI, ALS, APS, OLCF, ...).
Taxonomy: facility user-proposals COUNT as awards (@kyle approved 2026-08-06).

**Award numbers are FACILITY-SCOPED** — 703 numbers recur at >1 facility (same bare number,
different grants). The entity id hash is therefore salted with the facility code (`scheme`),
while `funder_award_id` keeps the bare cited form so the fold's generic normalized key matches
publisher-deposited bare numbers (measured 2026-08-06: 16,780 of 17,623 deposited DOE
bare-number ids = 95.2% hit this registry). For the 703 ambiguous numbers the alias election
picks one facility record — accepted, documented caveat. Facility code also rides in
`funder_scheme`; the Crossref grant DOI is carried in `doi`/`landing_page_url`.

No amounts published (§6.7 waiver). PI on 14,255 rows; award year only where the DOI suffix
carries it (7,053 rows, `proj.YYYY` pattern); no dates otherwise.

Parquet: `s3://openalex-ingest/awards/osti_facility/osti_facility_grants.parquet`
Upload (Rohan): `aws s3 cp ~/grants-work/next-batch/osti_facility_grants.parquet s3://openalex-ingest/awards/osti_facility/`

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.osti_facility_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/osti_facility/osti_facility_grants.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_projects FROM openalex.awards.osti_facility_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.osti_facility_raw LIMIT 5;

## Step 1.5: Funder-exists guard

In [ ]:
%sql
-- guard: fail loud if funder_id is absent from openalex.common.funder
-- (otherwise the Step 2 CROSS JOIN silently yields 0 rows)
SELECT assert_true((SELECT COUNT(*) FROM openalex.common.funder WHERE funder_id = 4320306084) = 1, 'funder_id 4320306084 missing from openalex.common.funder');

## Step 2: Create DOE facility Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.osti_facility_awards
USING delta
AS
WITH
fndr AS (
    SELECT CAST(funder_id AS BIGINT) as funder_id, display_name, ror_id, doi
    FROM openalex.common.funder WHERE funder_id = 4320306084
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id), ':', LOWER(COALESCE(g.scheme,''))))) % 9000000000 as id,  -- facility-salted: 703 award numbers recur at >1 facility (see header)
        COALESCE(NULLIF(TRIM(g.title), ''), CONCAT('DOE facility grant ', g.funder_award_id)) as display_name,
        g.description as description,
        f.funder_id,
        g.funder_award_id as funder_award_id,
        CASE WHEN TRY_CAST(g.amount AS DECIMAL(18,2)) > 0 THEN TRY_CAST(g.amount AS DECIMAL(18,2)) ELSE NULL END as amount,
        CASE WHEN TRY_CAST(g.amount AS DECIMAL(18,2)) > 0 THEN g.currency ELSE NULL END as currency,
        struct(CONCAT('https://openalex.org/F', f.funder_id) as id, f.display_name, f.ror_id, f.doi) as funder,
        'grant' as funding_type,
        g.scheme as funder_scheme,
        'osti_facility' as provenance,
        CAST(NULL AS DATE) as start_date,
        CAST(NULL AS DATE) as end_date,
        YEAR(TRY_TO_DATE(g.start_date_raw, 'yyyy')) as start_year,
        YEAR(TRY_TO_DATE(g.end_date_raw, 'yyyy')) as end_year,
        CASE
            WHEN g.pi_family IS NOT NULL THEN
                struct(g.pi_given as given_name, g.pi_family as family_name, CAST(NULL AS STRING) as orcid, CAST(NULL AS DATE) as role_start,
                    struct(g.institution as name, 'United States' as country, CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids) as affiliation)
            WHEN g.institution IS NOT NULL THEN
                struct(CAST(NULL AS STRING) as given_name, CAST(NULL AS STRING) as family_name, CAST(NULL AS STRING) as orcid, CAST(NULL AS DATE) as role_start,
                    struct(g.institution as name, 'United States' as country, CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids) as affiliation)
            ELSE NULL
        END as lead_investigator,
        CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE, affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE, affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>) as investigators,
        g.landing_page_url,
        g.doi as doi,  -- Crossref grant DOI (member 960 registration)
        CAST(NULL AS STRING) as works_api_url,
        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM openalex.awards.osti_facility_raw g
    CROSS JOIN fndr f
    WHERE g.funder_award_id IS NOT NULL
)
SELECT * FROM awards_transformed;

## Step 3: Insert into openalex_awards_raw

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw WHERE provenance = 'osti_facility' AND priority = 455;
INSERT INTO openalex.awards.openalex_awards_raw
SELECT id, display_name, description, funder_id, funder_award_id, amount, currency, funder, funding_type, funder_scheme, provenance, start_date, end_date, start_year, end_year, lead_investigator, co_lead_investigator, investigators, landing_page_url, doi, works_api_url, created_date, updated_date, 455 as priority
FROM openalex.awards.osti_facility_awards;

## Verification

In [ ]:
%sql
SELECT COUNT(*) total, COUNT(DISTINCT funder_award_id) uniq_award, COUNT(DISTINCT id) uniq_id, COUNT(DISTINCT funder_id) funders,
  SUM(CASE WHEN display_name IS NULL OR LENGTH(TRIM(display_name))=0 THEN 1 ELSE 0 END) blank,
  COUNT(amount) has_amount, COUNT(lead_investigator.given_name) has_person, COUNT(lead_investigator.affiliation.name) has_inst,
  COUNT(start_year) has_year, ROUND(SUM(amount)/1000000,1) total_m, MIN(start_year) min_yr, MAX(start_year) max_yr
FROM openalex.awards.osti_facility_awards;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) cnt, ROUND(SUM(amount)/1000000,1) total_m
FROM openalex.awards.osti_facility_awards GROUP BY 1 ORDER BY 2 DESC LIMIT 12;

In [ ]:
%sql
SELECT COUNT(*) as in_raw FROM openalex.awards.openalex_awards_raw WHERE provenance='osti_facility' AND priority=455;

## §6.5 Net-new go/no-go (post-fold dedup preview)

In [ ]:
%sql
-- §6.5 net-new go/no-go: how many rows survive the fold's lowest-priority-wins dedup.
-- Join the DISTINCT lower-priority id set: crossref sits at BOTH priority 1 and 2, so a
-- plain join fans out and inflates source_rows/shadowed (net_new stays correct).
-- Read: low pct_net_new => mostly already crossref-covered; question whether worth shipping.
SELECT
  COUNT(*) AS source_rows,
  COUNT_IF(x.id IS NULL) AS net_new_rows,
  COUNT_IF(x.id IS NOT NULL) AS shadowed_by_higher_priority,
  ROUND(try_divide(COUNT_IF(x.id IS NULL), COUNT(*)) * 100, 1) AS pct_net_new
FROM openalex.awards.osti_facility_awards a
LEFT JOIN (SELECT DISTINCT id FROM openalex.awards.openalex_awards_raw WHERE priority < 455) x
  ON x.id = a.id;